In [0]:
# %run ../init_framework_gold

In [0]:
# --- Catalog ---
dbutils.widgets.text("catalog_name", "lending_risk_dev") # Default value for manual testing
CATALOG = dbutils.widgets.get("catalog_name")

print(f"Running setup for Catalog: {CATALOG}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.calc_final_loan_score AS

WITH calculated_scores AS (
    SELECT 
        f.member_id,
        
        -- Extracting raw points for BI transparency 
        (p.last_payment_pts + p.total_payment_pts) AS raw_payment_pts,
        (d.delinq_pts + d.public_records_pts + d.public_bankruptcies_pts + d.inquiry_pts) AS raw_defaulter_pts,
        (f.loan_status_pts + f.home_pts + f.credit_limit_pts + f.grade_pts) AS raw_financial_pts,

        -- Final Weighted Algorithm
        (
            ((p.last_payment_pts + p.total_payment_pts) * 0.20) +
            ((d.delinq_pts + d.public_records_pts + d.public_bankruptcies_pts + d.inquiry_pts) * 0.45) +
            ((f.loan_status_pts + f.home_pts + f.credit_limit_pts + f.grade_pts) * 0.35)
        ) AS final_risk_score,

        -- Capture latest update time natively
        GREATEST(p._updated_at, d._updated_at, f._updated_at) AS score_updated_at

    FROM {CATALOG}.gold.calc_financial_performance f
    INNER JOIN {CATALOG}.gold.calc_payment_performance p 
        ON f.member_id = p.member_id
    INNER JOIN {CATALOG}.gold.calc_defaulters_performance d 
        ON f.member_id = d.member_id
)
SELECT 
    cs.*,
    -- Dynamic Grade Assignment via Lookup Join
    COALESCE(g.grade_name, 'F') AS risk_grade,
    COALESCE(g.description, 'Unacceptable') AS grade_description
FROM calculated_scores cs
LEFT JOIN {CATALOG}.silver.ref_loan_grading_scale g
    ON cs.final_risk_score >= g.min_score 
    AND cs.final_risk_score <= g.max_score;
    """
)